# 4. Deep Q-Learning (DQN)

En este cuaderno llegamos a la joya de la corona del Control con Aproximación de Funciones: **El Deep Q-Learning (DQN)**. A diferencia del método lineal con SARSA que vimos antes, aquí utilizaremos una **Red Neuronal Profunda** real con múltiples capas intermedias que le permite descubrir características no lineales del estado por sí sola.

Evaluaremos nuestro agente DQN en el entorno continuo clásico `CartPole-v1`, donde un coche intenta balancear un palo largo que tiene encima moviéndose de izquierda a derecha.

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import warnings
warnings.filterwarnings('ignore')

# Herramientas del proyecto
from src.agents.agent import Agent
from src.policies.epsilon_greedy import EpsilonGreedyPolicy
from src.learners.dqn import DQN
from src.plotting.plotting import plot_rewards, plot_episode_lengths

## Creación del Entorno
`CartPole-v1` devuelve un estado temporal continuo de 4 dimensiones `[posición del coche, velocidad del coche, ángulo del palo, velocidad angular del palo]`.

In [ ]:
env = gym.make("CartPole-v1")

state_size = env.observation_space.shape[0] # 4 dimensiones continuas
action_size = env.action_space.n # 2 dimensiones: (0: Izq, 1: Der)

print(f"Dimensión de estados continuos: {state_size}")
print(f"Número de acciones: {action_size}")
print(f"Device detectado: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## Parámetros Híper y Entrenamiento DQN
El entrenamiento profundo es un poco más lento e inestable al principio debido a que la red empieza aleatoria, pero converge en una política altamente adaptativa tras explorar lo suficiente usando el Experience Replay.

*Nota: Hemos acortado los episodios para demostrar la infraestructura rápidamente, pero en problemas reales DQN suele entrenarse entre miles y miles de episodios.*

In [ ]:
n_episodes = 250
n_runs = 2 # Múltiples ejecuciones garantizan la media si una red empieza atascada

# Hiperparámetros base
gamma = 0.99
learning_rate = 1e-3

# Componentes
policy = EpsilonGreedyPolicy(epsilon_start=1.0, epsilon_min=0.01, epsilon_decay=0.99)
dqn_learner = DQN(state_size=state_size, 
                  action_size=action_size, 
                  learning_rate=learning_rate, 
                  gamma=gamma,
                  buffer_capacity=5000, 
                  batch_size=64, 
                  target_update_freq=50)

print("Entrenando Agente DQN en CartPole-v1... (Esto tardará unos segundos/minutos)")
agent = Agent(env, dqn_learner, policy)
_, rewards, lengths, stats = agent.train(num_episodes=n_episodes, n_runs=n_runs)

## Visualización del Desempeño
La gráfica $f(t) = len(episodio_t)$ se vuelve fundamental aquí (y es un requisito original del análisis en *entornos complejos* propuesto en la rúbrica). 

En el caso del CartPole, **mayor duración del episodio significa mejor desempeño**. Si el episodio dura más pasos, quiere decir que el agente logró mantener el palo en equilibrio (puntuación máxima 500).

In [ ]:
# Aquí el número de pasos es equivalente directamente a la recompensa,
# ya que Cartpole da +1 reward por cada paso que el palo siga de pie.
plot_episode_lengths([lengths], legend_labels=["Deep Q-Learning (PyTorch)"], log_scale=False)
plot_rewards([rewards], legend_labels=["Deep Q-Learning (PyTorch)"], log_scale=False)